# Kvasir inference — Mean Teacher (Student + Teacher RGB / RGB-D)

Run inference on the Kvasir test set using trained **student** and **teacher** checkpoints. Saves segmentation masks for:
- **stu**: student (RGB-only) output
- **tea_rgb**: teacher RGB-only branch output
- **tea_rgbd**: teacher RGB-D fused output

Use the **Arguments** cell below to set checkpoint paths and output directory (or pass via CLI when run with `papermill`).

In [1]:
import sys
import os
import argparse
import torch
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms

from engine.Config import Config
from utils.common import get_proper_device
from test.eval import ImageFolderDataset
from data.transform import Resize, ToTensor
import models

## Arguments

Parse CLI args. When running interactively, defaults are used unless you set `sys.argv` in the previous run (e.g. `sys.argv = ['', '--stu_ckpt', 'save_dir/depth_enhance_mt_epoch105.pth']`).

In [ ]:
def parse_args():
    p = argparse.ArgumentParser(description='Kvasir inference: save stu, tea_rgb, tea_rgbd masks')
    p.add_argument('--stu_ckpt', type=str, default='save_dir/depth_enhance_mt_epoch105.pth',
                   help='Path to student checkpoint (.pth)')
    p.add_argument('--tea_ckpt', type=str, default='save_dir/depth_enhance_mt_teacher_epoch105.pth',
                   help='Path to teacher checkpoint (.pth)')
    p.add_argument('--out_dir', type=str, default='kvasir_inference',
                   help='Directory to save predicted masks (subdirs: stu, tea_rgb, tea_rgbd)')
    p.add_argument('--dataset_root', type=str, default=None,
                   help='Kvasir test root (default: from config data.test.dataset_root)')
    p.add_argument('--config', type=str, default='cfg/depth_enhance_mt.yaml',
                   help='Config YAML for model names and test data paths')
    p.add_argument('--batch_size', type=int, default=1, help='Inference batch size')
    p.add_argument('--device', type=str, default=None, help='Device (default: from config)')
    return p.parse_args()

# In notebook, parse_args() may see kernel args; pass [] to use defaults if you did not set sys.argv
if 'ipykernel' in sys.modules and not any(x.startswith('--stu_ckpt') or x.startswith('--tea_ckpt') for x in sys.argv):
    args = parse_args()  # use defaults when no checkpoint args passed
else:
    args = parse_args()

print('Student checkpoint:', args.stu_ckpt)
print('Teacher checkpoint:', args.tea_ckpt)
print('Output directory:', args.out_dir)

In [ ]:
cfg = Config(config_file=args.config)
device = get_proper_device(args.device or cfg.get('device'))
num_classes = cfg.get('model.num_channels_output', 1)
dataset_root = args.dataset_root or cfg.get('data.test.dataset_root')
resize_h = cfg.get('data.test.resize_height', 320)
resize_w = cfg.get('data.test.resize_width', 320)
image_dir = cfg.get('data.test.image_dirname', 'images')
mask_dir = cfg.get('data.test.mask_dirname', 'masks')
depth_dir = str(cfg.get('data.test.depth_dirname', 'depth-v1'))

val_test_transform = transforms.Compose([Resize((resize_w, resize_h)), ToTensor()])
eval_data = ImageFolderDataset(
    dataset_root=dataset_root,
    image_dirname=image_dir,
    mask_dirname=mask_dir,
    depth_dirname=depth_dir,
    transform=val_test_transform,
    list_name=None,
)
eval_loader = DataLoader(eval_data, batch_size=args.batch_size, shuffle=False, num_workers=0)
print(f'Dataset: {dataset_root} | samples: {len(eval_data)}')

In [ ]:
stu_model = getattr(models, cfg.get('model.stu_model.name'))(num_classes=num_classes).to(device)
tea_model = getattr(models, cfg.get('model.tea_model.name'))(num_classes=num_classes).to(device)

stu_model.load_state_dict(torch.load(args.stu_ckpt, map_location=device), strict=True)
tea_model.load_state_dict(torch.load(args.tea_ckpt, map_location=device), strict=True)
stu_model.eval()
tea_model.eval()
print('Models loaded.')

In [ ]:
os.makedirs(args.out_dir, exist_ok=True)
for sub in ('stu', 'tea_rgb', 'tea_rgbd'):
    os.makedirs(os.path.join(args.out_dir, sub), exist_ok=True)

def save_mask(tensor, path):
    """tensor: (H,W) or (1,H,W) in [0,1]; save as 0/255 PNG."""
    if tensor.dim() == 3:
        tensor = tensor.squeeze(0)
    arr = (tensor.cpu().numpy() >= 0.5).astype(np.uint8) * 255
    Image.fromarray(arr, mode='L').save(path)

with torch.no_grad():
    for batch in eval_loader:
        img = batch['image'].to(device)
        depth = batch['depth'].to(device)
        filenames = batch['filename']
        if isinstance(filenames, str):
            filenames = [filenames]
        else:
            filenames = list(filenames)

        stu_out = stu_model(img)
        tea_out = tea_model(img, depth)
        tea_rgb = tea_out['rgb']
        tea_rgbd = tea_out['rgb_depth']

        for i in range(img.size(0)):
            base = os.path.splitext(filenames[i])[0]
            save_mask(stu_out[i], os.path.join(args.out_dir, 'stu', base + '.png'))
            save_mask(tea_rgb[i], os.path.join(args.out_dir, 'tea_rgb', base + '.png'))
            save_mask(tea_rgbd[i], os.path.join(args.out_dir, 'tea_rgbd', base + '.png'))

print(f'Done. Masks saved under {args.out_dir}/stu, {args.out_dir}/tea_rgb, {args.out_dir}/tea_rgbd')